# 📡 Traffic & Marketing Channel Analysis
> **Author:** Binh Pham | Maven Fuzzy Factory

Deep dive into where sessions come from, conversion rates, device split, and repeat visitor behaviour.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5),
                     'axes.titlesize': 13, 'axes.titleweight': 'bold'})

BASE  = '..'
PROC  = f'{BASE}/data/processed'
sessions = pd.read_csv(f'{PROC}/cleaned_website_sessions.csv', parse_dates=['created_at'])
orders   = pd.read_csv(f'{PROC}/cleaned_orders.csv',           parse_dates=['created_at'])
master   = pd.read_csv(f'{PROC}/master_orders.csv',            parse_dates=['created_at'])


## 1. Session Volume by Channel

In [ ]:

ch = sessions.groupby(['utm_source','utm_campaign']).agg(
    sessions=('website_session_id','nunique'),
    repeat_sessions=('is_repeat_session','sum')
).reset_index()
ch['repeat_rate'] = (ch['repeat_sessions'] / ch['sessions'] * 100).round(1)
ch['channel'] = ch['utm_source'].str.title() + ' / ' + ch['utm_campaign'].str.title()
ch = ch.sort_values('sessions', ascending=True)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(ch['channel'], ch['sessions'], color=sns.color_palette('muted', len(ch)))
ax.bar_label(bars, labels=[f"{v:,}" for v in ch['sessions']], padding=5)
ax.set_xlabel('Sessions')
ax.set_title('Total Sessions by Traffic Channel')
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_sessions_by_channel.png', bbox_inches='tight')
plt.show()
ch[['channel','sessions','repeat_rate']].sort_values('sessions', ascending=False)


## 2. Session-to-Order Conversion Rate by Channel

In [ ]:

sess_agg  = sessions.groupby(['utm_source','utm_campaign'])['website_session_id'].nunique().reset_index(name='total_sessions')
order_agg = master.groupby(['utm_source','utm_campaign'])['order_id'].nunique().reset_index(name='orders')
cvr = sess_agg.merge(order_agg, on=['utm_source','utm_campaign'], how='left').fillna(0)
cvr['cvr_pct'] = (cvr['orders'] / cvr['total_sessions'] * 100).round(2)
cvr['channel'] = cvr['utm_source'].str.title() + ' / ' + cvr['utm_campaign'].str.title()
cvr = cvr.sort_values('cvr_pct', ascending=False)

fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#27ae60' if v > cvr['cvr_pct'].median() else '#e74c3c' for v in cvr['cvr_pct']]
bars = ax.bar(cvr['channel'], cvr['cvr_pct'], color=colors)
ax.bar_label(bars, labels=[f"{v:.1f}%" for v in cvr['cvr_pct']], padding=3)
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Session → Order Conversion Rate by Channel')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_cvr_by_channel.png', bbox_inches='tight')
plt.show()


## 3. Monthly Session Trend

In [ ]:

monthly = sessions.groupby(['year_month','utm_source'])['website_session_id'].nunique().reset_index(name='sessions')
pivot   = monthly.pivot(index='year_month', columns='utm_source', values='sessions').fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
pivot.plot(ax=ax, marker='o', linewidth=2)
ax.set_title('Monthly Sessions by Traffic Source')
ax.set_xlabel('Month')
ax.set_ylabel('Sessions')
ax.legend(title='Source', bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_monthly_sessions.png', bbox_inches='tight')
plt.show()


## 4. Device Type Split

In [ ]:

device = sessions.groupby('device_type')['website_session_id'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].pie(device.values, labels=device.index, autopct='%1.1f%%',
            colors=['#3498db','#e67e22'], startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('Session Share by Device Type')

# CVR by device
dev_cvr = sessions.merge(orders[['website_session_id','order_id']], on='website_session_id', how='left')
dev_cvr = dev_cvr.groupby('device_type').agg(
    sessions=('website_session_id','nunique'), orders=('order_id','nunique')
).reset_index()
dev_cvr['cvr'] = (dev_cvr['orders'] / dev_cvr['sessions'] * 100).round(2)
axes[1].bar(dev_cvr['device_type'], dev_cvr['cvr'], color=['#3498db','#e67e22'])
for i, row in dev_cvr.iterrows():
    axes[1].text(i, row['cvr'] + 0.1, f"{row['cvr']:.2f}%", ha='center', fontweight='bold')
axes[1].set_ylabel('Conversion Rate (%)')
axes[1].set_title('Conversion Rate by Device Type')

plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_device_split.png', bbox_inches='tight')
plt.show()


## 5. New vs Repeat Visitor Revenue

In [ ]:

visitor = master.groupby('is_repeat_session').agg(
    orders=('order_id','nunique'),
    revenue=('price_usd','sum'),
    avg_order_val=('price_usd','mean')
).round(2).reset_index()
visitor['label'] = visitor['is_repeat_session'].map({0:'New Visitor', 1:'Repeat Visitor'})

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = [('orders','Orders'), ('revenue','Revenue ($)'), ('avg_order_val','Avg Order Value ($)')]
colors  = ['#9b59b6', '#1abc9c']
for ax, (col, title) in zip(axes, metrics):
    ax.bar(visitor['label'], visitor[col], color=colors)
    for i, v in enumerate(visitor[col]):
        ax.text(i, v + visitor[col].max()*0.01, f"{v:,.0f}", ha='center', fontweight='bold')
    ax.set_title(title)

fig.suptitle('New vs Repeat Visitor Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_new_vs_repeat.png', bbox_inches='tight')
plt.show()
